## AuxTel Tracking tests

To measure elevation motor torque as a function of elevation.

In [1]:
import sys, time, os, asyncio

from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from lsst.ts import salobj
from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from astropy.time import Time, TimeDelta
from astropy.coordinates import AltAz, ICRS, EarthLocation, Angle, FK5
import astropy.units as u

/opt/lsst/software/stack/conda/miniconda3-py38_4.9.2/envs/lsst-scipipe-0.4.1/lib/python3.8/site-packages/jose/backends/cryptography_backend.py:23: CryptographyDeprecationWarning: int_from_bytes is deprecated, use int.from_bytes instead
  from cryptography.utils import int_from_bytes, int_to_bytes


In [2]:
# Set Cerro Pachon location
location = EarthLocation.from_geodetic(lon=-70.747698*u.deg,
                                       lat=-30.244728*u.deg,
                                       height=2663.0*u.m)

In [3]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [4]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [5]:
#get classes and start them
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.00 sec
Read 1 history items for RemoteEvent(ATArchiver, 0, authList)
Read 1 history items for RemoteEvent(ATArchiver, 0, errorCode)
Read 100 history items for RemoteEvent(ATArchiver, 0, heartbeat)
Read 100 history items for RemoteEvent(ATArchiver, 0, imageInOODS)
Read 100 history items for RemoteEvent(ATArchiver, 0, imageRetrievalForArchiving)
Read 1 history items for RemoteEvent(ATArchiver, 0, logLevel)
Read 1 history items for RemoteEvent(ATArchiver, 0, logMessage)
Read 1 history items for RemoteEvent(ATArchiver, 0, simulationMode)
Read 1 history items for RemoteEvent(ATArchiver, 0, softwareVersi

[[None, None, None, None, None, None, None], [None, None, None, None]]

trajectory DDS read queue is filling: 39 of 100 elements
m1AirPressure DDS read queue is filling: 22 of 100 elements
loadCell DDS read queue is filling: 22 of 100 elements
mount_positions DDS read queue is filling: 71 of 100 elements
torqueDemand DDS read queue is filling: 39 of 100 elements
mountStatus DDS read queue is full (100 elements); data may be lost
nasymth_m3_mountMotorEncoders DDS read queue is filling: 39 of 100 elements
guidingAndOffsets DDS read queue is full (100 elements); data may be lost
mount_Nasmyth_Encoders DDS read queue is filling: 39 of 100 elements
currentTargetStatus DDS read queue is full (100 elements); data may be lost
mount_AzEl_Encoders DDS read queue is filling: 40 of 100 elements
mount_AzEl_Encoders python read queue is filling: 39 of 100 elements
measuredTorque DDS read queue is filling: 40 of 100 elements
measuredMotorVelocity DDS read queue is filling: 40 of 100 elements
azEl_mountMotorEncoders DDS read queue is filling: 40 of 100 elements


In [ ]:
# enable components if required
# Failed
await atcs.enable()
await latiss.enable()

In [ ]:
await latiss.rem.atcamera.cmd_start.set_start(timeout=30)

In [ ]:
await salobj.set_summary_state(latiss.rem.atcamera, salobj.State.ENABLED)

In [ ]:
await latiss.rem.atarchiver.cmd_start.set_start(timeout=30)

In [ ]:
await salobj.set_summary_state(latiss.rem.atarchiver, salobj.State.ENABLED)

In [ ]:
# Everything now enabled 11:45 AM

In [6]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = False
atcs.check.atdometrajectory = False

In [7]:
# turn on ATAOS corrections just to make sure the mirror is under air
tmp = await atcs.rem.ataos.cmd_enableCorrection.set_start(m1=True, hexapod=True, atspectrograph=False)

In [8]:
# Ensure we're using Nasmyth 2
await atcs.rem.atptg.cmd_focusName.set_start(focus=3)

AckError: msg='Command failed', ackcmd=(ackcmd private_seqNum=634413789, ack=<SalRetCode.CMD_FAILED: -302>, error=6612, result='Rejected : command not allowed in current state')

In [ ]:
# point telescope to desired starting position
start_az=90
start_el=80
start_rot_pa=0
await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)

In [ ]:
# now loop from elevation of +80 down to +20 in 5 degree increments
# Should be images 2 - 14
start_az=90
start_el=80
start_rot_pa=0

for i in range(13):
    start_el = 80 - i * 5
    # point telescope to desired starting position
    await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)
    await asyncio.sleep(2)
    # take a 2s dark
    await latiss.take_darks(2.0, 1)


In [ ]:
# For shutdown of system
await atcs.stop_tracking()

In [ ]:
# turn off corrections
tmp = await atcs.rem.ataos.cmd_disableCorrection.set_start(m1=True, hexapod=True, atspectrograph=True)

In [ ]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [ ]:
# Putting everything back in standby.
# Failed as it usually does
await atcs.shutdown()

In [ ]:
await atcs.rem.atdome.cmd_start.set_start(settingsToApply="test", timeout=30)

In [ ]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.STANDBY, settingsToApply="test")

In [ ]:
await salobj.set_summary_state(latiss.rem.atspectrograph, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atcamera, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atheaderservice, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atarchiver, salobj.State.STANDBY)

In [ ]:
# Everything back in standby